# POC segmentation CT cardiaque — exécution dans Colab

Ce notebook exécute les scripts du dépôt GitHub sur le GPU de Colab. Les données vont dans ton Google Drive.

**Avant de commencer** : menu **Exécution → Modifier le type d'exécution → GPU T4**. Puis exécute les cellules une par une (▶).

## 1. Relier Drive et vérifier le GPU
Une fenêtre d'autorisation s'ouvre : accepte avec ton **compte Google personnel**. Tu dois voir **Tesla T4**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Récupérer le dépôt GitHub
Le dépôt GitHub `Cameronprr/poc-segmentation-ct` est copié dans Colab (cette copie disparaît à la fin de la session ; le dépôt sur GitHub reste la référence).

In [ ]:
%cd /content
# On sort d'abord de l'ancien dossier du dépôt : sinon, après sa suppression, le dossier courant
# n'existe plus et le shell affiche l'erreur « getcwd ».
PSEUDO = 'Cameronprr'
!rm -rf /content/poc-segmentation-ct
!git clone https://github.com/{PSEUDO}/poc-segmentation-ct.git /content/poc-segmentation-ct
%cd /content/poc-segmentation-ct
!ls

## 3. Installer les bibliothèques de l'étape
Pour l'étape 2, seule `nibabel` (lecture des fichiers NIfTI) est nécessaire. Les autres (MONAI, nnU-Net, TotalSegmentator) seront installées aux étapes suivantes.

In [ ]:
!pip -q install nibabel

## 4. Étape 2 — Télécharger MM-WHS dans Drive
À ne faire qu'une fois : les données restent ensuite dans ton Drive. `megatools` est un petit outil qui télécharge depuis le service MEGA. Compte plusieurs minutes.

In [ ]:
DONNEES = '/content/drive/MyDrive/poc_ct/mmwhs'
!mkdir -p "$DONNEES"
!apt-get -qq install megatools > /dev/null
!megadl --path "$DONNEES" "https://mega.nz/folder/UNMF2YYI#1cqJVzo4p_wESv9P_pc8uA"

# Décompresser d'éventuelles archives
import glob, zipfile, os
for z in glob.glob(DONNEES + '/**/*.zip', recursive=True):
    print('Décompression :', z)
    zipfile.ZipFile(z).extractall(os.path.dirname(z))

## 5. Étape 2 — Vérifier les données
Le script `scripts/02_verifier_donnees.py` contrôle que les 20 cas du découpage figé sont complets et affiche leurs caractéristiques. **Résultat attendu : CONFORME — 20 cas complets.**

In [ ]:
!python scripts/02_verifier_donnees.py --donnees "$DONNEES"

**Fin de l'étape 2.** Copie le tableau affiché (ou le message d'erreur) et envoie-le dans la conversation.

## 6. Étape 3 — Installer MONAI
MONAI est une bibliothèque d'imagerie médicale (chargement des scanners, transformations, métriques). Le script de l'étape 3 s'en sert pour lire les volumes et calculer les miniatures. À refaire à chaque nouvelle session Colab, comme les cellules 1 et 2.

In [ ]:
!pip -q install "monai>=1.3" nibabel

## 7. Étape 3 — Contrôle qualité détaillé et conversion au format nnU-Net
Le script `scripts/03_preparer.py` contrôle les 20 cas : forme, affine, espacement, orientation, étiquettes et doublons entre test et entraînement. Il écrit ensuite le rapport `rapport_qc_etape3.csv`. Si tout est conforme, il écrit aussi le jeu `Dataset101_MMWHS` dans `poc_ct/nnunet_raw`.

**Résultat attendu :** « Résultat du contrôle : CONFORME », puis 15 cas dans imagesTr/labelsTr et 5 dans imagesTs/labelsTs.

Si le script s'arrête, le dernier message indique pourquoi :
- **NON CONFORME** : rien n'a été écrit au format nnU-Net ;
- **existe déjà** : le jeu a déjà été écrit. Ne relance avec `--ecraser` que si tu veux vraiment le remplacer ;
- **ERREUR PENDANT LA CONVERSION** : le dossier est incomplet. Relance avec `--ecraser` une fois la cause corrigée.

Les lignes « QUASI-DOUBLON possible » ne bloquent pas : envoie-les dans la conversation pour qu'on les examine.

In [ ]:
DONNEES = '/content/drive/MyDrive/poc_ct/mmwhs'
SORTIE = '/content/drive/MyDrive/poc_ct/nnunet_raw'
!python scripts/03_preparer.py --donnees "$DONNEES" --sortie "$SORTIE"

### 7 bis. Reconstruire `Dataset101_MMWHS` avec le nouveau découpage (décision D20)
**Avant :** relance les sections 1, 2 et 6. La section 2 récupère le dépôt à jour, avec le nouveau `splits/mmwhs_split.json`. Sinon, le script reconstruirait l'**ancien** découpage sans rien signaler. Pour vérifier, `!head -12 splits/mmwhs_split.json` doit montrer en test 1004, 1010, 1011, 1013 et 1014.

Le découpage figé a été remplacé une fois (D20) : le jeu nnU-Net déjà écrit dans Drive suit l'**ancien** découpage. Cette variante relance le même script avec `--ecraser` : l'ancien dossier `Dataset101_MMWHS` est **supprimé**, puis réécrit avec le découpage de `splits/mmwhs_split.json`. Le rapport `rapport_qc_etape3.csv` est aussi remplacé.

**À n'exécuter qu'une fois, maintenant.** N'exécute pas la cellule 7 juste au-dessus : elle s'arrêterait sur « existe déjà ». Une exception : si le script affiche **ERREUR PENDANT LA CONVERSION**, le dossier est incomplet. Corrige la cause, puis relance cette même cellule.

**Résultat attendu :**
- « Résultat du contrôle : CONFORME » ;
- en test `mmwhs_1004`, `1010`, `1011`, `1013` et `1014` (5 cas dans imagesTs/labelsTs), et 15 cas dans imagesTr/labelsTr ;
- la liste des paires test/entraînement de corrélation supérieure à 0,95, à m'envoyer pour qu'on vérifie qu'aucune n'est suspecte.

Si le script s'arrête sur une règle D20 (patient à plusieurs examens en test), rien n'a été modifié dans Drive. S'il affiche NON CONFORME, `Dataset101_MMWHS` n'est pas touché, mais `rapport_qc_etape3.csv` est quand même remplacé.

In [ ]:
DONNEES = '/content/drive/MyDrive/poc_ct/mmwhs'
SORTIE = '/content/drive/MyDrive/poc_ct/nnunet_raw'
!python scripts/03_preparer.py --donnees "$DONNEES" --sortie "$SORTIE" --ecraser

**Fin du script 03.** Copie tout ce que le script a affiché (tableau, ressemblances, avertissements, résultat) et envoie-le dans la conversation pour la vérification par l'agent QA.

## 8. Étape 3 — Examiner à l'œil les deux quasi-doublons possibles
**Examen fait le 8 novembre 2026, conclusion dans D20.** Depuis la reconstruction (section 7 bis), 1005 et 1020 sont en `imagesTr` : cette cellule afficherait « Fichier introuvable ». **Ne plus la relancer.**

Le script 03 a signalé deux paires (scanner de test / scanner d'entraînement) qui ont exactement la même taille et le même espacement, avec des miniatures très corrélées :
- `mmwhs_1005` (test) / `mmwhs_1012` (entraînement), corrélation 0,962 ;
- `mmwhs_1020` (test) / `mmwhs_1001` (entraînement), corrélation 0,974.

Si l'une de ces paires était **le même examen** (ou le même patient), le modèle aurait « vu » pendant l'entraînement un cas très proche du lot de test, et le verdict final serait trop optimiste. Mais attention : même taille et même espacement ne prouvent rien (beaucoup de scanners ont ces réglages), et **une corrélation de 0,96-0,97 entre miniatures est attendue entre deux thorax différents cadrés de la même façon**. Ce n'est pas un signal fort en soi : il faut regarder.

**Vocabulaire**
- **Coupe axiale** : une tranche horizontale du corps (comme une tranche de saucisson), vue depuis les pieds du patient. On y voit le cœur, les côtes en arc, le sternum devant et la colonne vertébrale derrière.
- **Coupe coronale** : une tranche verticale, vue de face (le patient debout face à toi). Souvent plus parlante pour reconnaître une silhouette : forme du thorax, du cœur, des poumons, hauteur du diaphragme.
- **Coupe du milieu** : la tranche située à mi-hauteur (axiale) ou à mi-profondeur (coronale) du volume.
- **Sens d'affichage** : celui des radiologues. Sur les deux vues, **la droite du patient est à gauche de l'image** (comme quand on fait face à quelqu'un). L'avant du patient est en haut de la coupe axiale, la tête en haut de la coupe coronale. Les deux scanners d'une paire sont affichés exactement dans le même sens.

**Ce qu'affiche la cellule** : pour chaque paire, une figure de 2 lignes (axiale en haut, coronale en bas) et 3 colonnes : scanner de test, scanner d'entraînement, puis leur **différence** (test − entraînement). Les deux scanners utilisent la même fenêtre d'affichage, de -200 à 800 HU (la même que le script 03).

**Comment lire la carte de différence** (blanc = 0 ; rouge = le test est plus clair ; bleu = l'entraînement est plus clair ; au-delà de ±500 HU la couleur sature) :
- **carte toute blanche** (au plus un léger bruit) : c'est **le même examen**, donc un doublon ;
- **carte non blanche** (contours en paires rouge/bleu) : ce sont des **examens différents**, mais **pas forcément des patients différents** : le même patient scanné un autre jour, ou à une autre phase du battement cardiaque, donnerait aussi des contours. Et un doublon simplement décalé d'une ou deux coupes, ou recadré, donnerait lui aussi des contours rouge/bleu : d'où le « meilleur décalage » ci-dessous.

**Pour juger « même patient »**, compare l'anatomie dans les deux premières colonnes : forme des vertèbres, des côtes et du sternum, calcifications (petites taches très blanches, par exemple sur l'aorte ou les coronaires), forme générale du thorax, taille du cœur.

**Les chiffres affichés sous chaque figure**
- **forme, espacement, orientation, origine** : l'origine est la position dans l'espace (en mm) du premier voxel du volume, lue dans l'en-tête du fichier. Deux examens distincts ont souvent des origines différentes ; une origine identique au centième de mm est un indice de copie (pas une preuve).
- **différence absolue moyenne** (axiale, coronale) : moyenne de |test − entraînement| sur la coupe affichée, en HU. 0,0 veut dire des coupes identiques au voxel près **dans la fenêtre affichée** (-200 à 800 HU). C'est un repère : l'air autour du corps vaut -200 dans les deux images (différence nulle), ce qui fait baisser la moyenne.
- **meilleur décalage** : la coupe axiale du milieu du test est comparée aux 21 coupes de l'entraînement situées de 10 coupes en dessous à 10 coupes au-dessus. On affiche le décalage qui donne la plus petite différence, et cette différence. Un doublon décalé donnerait une différence proche de 0 pour un décalage non nul ; deux examens différents restent loin de 0 quel que soit le décalage.
- **volumes entiers identiques ?** : on calcule l'empreinte (SHA-256, une suite de 64 caractères tirée de tous les voxels) de chaque volume. « IDENTIQUES » = même volume au voxel près (doublon exact) ; « différents » = au moins un voxel diffère. Le script 03 a déjà fait ce contrôle ; il est refait ici pour avoir tous les éléments au même endroit.

**Avant de lancer** : après un redémarrage de session, relance la section 1 (Drive) ; nibabel et matplotlib sont préinstallés dans Colab. La cellule ne fait que **lire** deux scanners par paire, un à la fois : elle n'écrit aucun fichier, ne lit aucun masque (le dossier `labelsTs` reste fermé, décision D18) et ne touche pas au découpage figé.

Envoie ta conclusion dans la conversation ; la décision (garder, ou retirer un cas) sera tracée dans le journal.

In [ ]:
# Comparaison à l'œil des deux paires signalées par le script 03. LECTURE SEULE : rien n'est écrit.
# Seuls les scanners (images) sont lus : aucun masque, et surtout pas labelsTs (décision D18).
import hashlib, os
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt

DATASET = '/content/drive/MyDrive/poc_ct/nnunet_raw/Dataset101_MMWHS'
# (numéro du cas de test, numéro du cas d'entraînement, corrélation des miniatures donnée par le script 03)
PAIRES = [('1005', '1012', 0.962),
          ('1020', '1001', 0.974)]
FENETRE = (-200, 800)   # fenêtre d'affichage en HU, identique pour les deux scanners (comme le script 03)
ECHELLE_DIFF = 500      # carte de différence de -500 à +500 HU (au-delà, la couleur sature)
DECALAGE_MAX = 10       # recherche du meilleur décalage : coupes k-10 à k+10 de l'entraînement

def vue_radiologique(coupe):
    """Écrête la coupe dans la FENETRE et la met dans le sens des radiologues pour imshow(origin='lower') :
    np.flip sur le 1er axe = droite du patient à gauche de l'image ; .T = avant (ou tête) en haut."""
    return np.flip(np.clip(coupe.astype(np.float32), *FENETRE), 0).T

def lire(chemin, k_ref=None):
    """Lit UN scanner. Renvoie ses coupes du milieu (axiale, coronale), quelques infos, et, si k_ref est
    donné, les coupes axiales k_ref-10..k_ref+10 (pour le meilleur décalage). Le volume complet n'existe
    que dans cette fonction : la mémoire est libérée à la sortie."""
    img = nib.load(chemin)
    # Remise en orientation RAS (1er axe vers la droite du patient, 2e vers l'avant, 3e vers le haut) :
    # ainsi les deux scanners d'une paire sont toujours traités dans le même sens.
    ras = nib.as_closest_canonical(img)
    vol = np.asanyarray(ras.dataobj)                              # voxels en HU
    h = hashlib.sha256(f'{vol.shape}{vol.dtype}'.encode())        # empreinte de TOUT le volume
    h.update(np.ascontiguousarray(vol).tobytes())
    sx, sy, sz = np.sqrt((ras.affine[:3, :3] ** 2).sum(axis=0))   # espacement en mm le long de chaque axe
    infos = {'forme': img.shape,
             'espacement': tuple(round(float(z), 3) for z in img.header.get_zooms()[:3]),
             'orientation': ''.join(nib.aff2axcodes(img.affine)),
             'origine': tuple(round(float(v), 2) for v in img.affine[:3, 3]),   # position du 1er voxel (mm)
             'empreinte': h.hexdigest()}
    k, j = vol.shape[2] // 2, vol.shape[1] // 2   # milieu en hauteur (axiale), milieu avant-arrière (coronale)
    # aspect = rapport des espacements, pour ne pas écraser l'image si les voxels ne sont pas cubiques.
    vues = {'axiale':   (vue_radiologique(vol[:, :, k]), k, vol.shape[2], sy / sx),
            'coronale': (vue_radiologique(vol[:, j, :]), j, vol.shape[1], sz / sx)}
    voisines = {}
    if k_ref is not None:
        for d in range(-DECALAGE_MAX, DECALAGE_MAX + 1):
            if 0 <= k_ref + d < vol.shape[2]:
                voisines[d] = vue_radiologique(vol[:, :, k_ref + d])
    del vol, ras, img
    return vues, infos, voisines

for test, entr, r in PAIRES:
    ch_test = f'{DATASET}/imagesTs/mmwhs_{test}_0000.nii.gz'
    ch_entr = f'{DATASET}/imagesTr/mmwhs_{entr}_0000.nii.gz'
    manquants = [c for c in (ch_test, ch_entr) if not os.path.exists(c)]
    if manquants:
        print('Fichier introuvable (Drive est-il relié ? section 1) :', manquants)
        continue
    v_t, i_t, _ = lire(ch_test)                                   # un volume à la fois
    v_e, i_e, voisines = lire(ch_entr, k_ref=v_t['axiale'][1])
    lus = [('TEST', f'mmwhs_{test}', v_t, i_t), ('entraînement', f'mmwhs_{entr}', v_e, i_e)]

    fig, axes = plt.subplots(2, 3, figsize=(16, 11))
    fig.suptitle(f'mmwhs_{test} (TEST) contre mmwhs_{entr} (entraînement) — corrélation des miniatures {r}'
                 '\n(droite du patient à gauche de l\'image)', fontsize=14)
    ecarts = {}
    for ligne, vue in enumerate(['axiale', 'coronale']):
        for col, (lot, nom, vues, _) in enumerate(lus):
            coupe, num, total, aspect = vues[vue]
            axes[ligne, col].imshow(coupe, cmap='gray', vmin=FENETRE[0], vmax=FENETRE[1],
                                    origin='lower', aspect=aspect)
            axes[ligne, col].set_title(f'{nom} — lot {lot}\ncoupe {vue} n°{num + 1} sur {total}')
        c_t, num_t, total_t, aspect_t = v_t[vue]                  # infos du TEST pour la carte de différence
        c_e, aspect_e = v_e[vue][0], v_e[vue][3]
        ax = axes[ligne, 2]
        if c_t.shape == c_e.shape and np.isclose(aspect_t, aspect_e):
            diff = c_t - c_e
            ecarts[vue] = float(np.abs(diff).mean())
            im = ax.imshow(diff, cmap='RdBu_r', vmin=-ECHELLE_DIFF, vmax=ECHELLE_DIFF,
                           origin='lower', aspect=aspect_t)       # RdBu_r : bleu < 0 < rouge, blanc = 0
            fig.colorbar(im, ax=ax, shrink=0.8, label='test − entraînement (HU)')
            ax.set_title(f'Différence test − entraînement\ncoupe {vue} n°{num_t + 1} sur {total_t}')
        else:
            ax.text(0.5, 0.5, 'Formes ou espacements différents :\ndifférence non calculable',
                    ha='center', va='center')
    for ax in axes.ravel():
        ax.axis('off')   # pas de graduations : seules les images comptent
    plt.tight_layout()
    plt.show()

    # Quelques chiffres simples sous la figure
    for lot, nom, _, infos in lus:
        print(f"{nom:<12} lot {lot:<13} forme {infos['forme']}  espacement {infos['espacement']} mm  "
              f"orientation {infos['orientation']}  origine {infos['origine']} mm")
    for vue, e in ecarts.items():
        print(f'Différence absolue moyenne, coupe {vue} (fenêtre {FENETRE[0]} à {FENETRE[1]} HU) : {e:.1f} HU')
    # Meilleur décalage : coupe axiale du milieu du test contre les coupes voisines de l'entraînement
    ref = v_t['axiale'][0]
    par_decalage = {d: float(np.abs(ref - c).mean()) for d, c in voisines.items() if c.shape == ref.shape}
    if par_decalage:
        d = min(par_decalage, key=par_decalage.get)
        print(f'Meilleur décalage (axiale, de -{DECALAGE_MAX} à +{DECALAGE_MAX} coupes) : {d:+d} coupe(s), '
              f'différence absolue moyenne {par_decalage[d]:.1f} HU')
    else:
        print('Meilleur décalage : non calculable (formes différentes)')
    print('Volumes entiers (empreinte SHA-256) :',
          'IDENTIQUES (doublon exact)' if i_t['empreinte'] == i_e['empreinte'] else 'différents')
    print()
    del lus, v_t, v_e, voisines

## 9. Découpage par patient (décision D20)
**Pourquoi.** L'examen de la section 8 a montré que les paires `1005`/`1012` et `1020`/`1001` ont la même forme, le même espacement et la même origine au centième de mm, mais des voxels différents : très probablement **le même patient scanné à deux phases du battement cardiaque** (cœur contracté ou relâché). Or `1005` et `1020` sont en test, `1012` et `1001` en entraînement. Un même patient des deux côtés fausse le verdict : le modèle aurait déjà « vu » ce patient. On refait donc le découpage **par patient** : le lot de test ne contient que des **patients à un seul examen** (5 cas = 5 patients indépendants, chaque patient compte une fois dans le verdict) ; les patients à plusieurs examens vont, entiers, en entraînement.

**Ce que fait le script `scripts/03b_regrouper_patients.py`.**
1. Il lit uniquement l'**en-tête** des 20 scanners (taille, espacement, position, orientation) : c'est instantané, aucun voxel n'est chargé et aucun masque n'est lu.
2. Il regroupe en « patients probables » les cas qui ont la même forme, le même espacement et la même **affine** (la matrice qui place le volume dans l'espace, origine comprise) à 0,01 mm près, et affiche tous les groupes.
3. Il vérifie que la position est **réellement enregistrée** dans l'en-tête : les colonnes `sform/qform` donnent deux codes qui le disent. S'ils valent tous les deux 0, il n'y a pas de position enregistrée : nibabel en fabrique une par défaut, identique pour tous les volumes de même taille. Le cas est alors marqué **« origine NON FIABLE »** et n'est regroupé avec aucun autre.
4. Il liste pour information les **paires proches** : même plan de coupe et même position gauche-droite et avant-arrière (à 1 mm près), mais un nombre de coupes ou une hauteur différents. Elles ne sont pas regroupées automatiquement.
5. Il tire au sort, avec la graine 42, 5 patients **à un seul examen** (et à origine fiable) pour le lot de test. Le tirage est **stratifié par épaisseur de coupe** (ici l'espacement entre deux coupes, lu dans l'en-tête) : 1 patient à 0,45 mm et 4 à 0,625 mm, pour que le test contienne les deux types d'examens. Tous les autres cas, dont les deux paires, vont en entraînement / validation croisée.
6. Il affiche la proposition, les cas qui changent de lot, et écrit le fichier `mmwhs_split_par_patient_PROPOSITION.json` dans `poc_ct` (Drive). **Le dépôt et `splits/mmwhs_split.json` ne sont pas modifiés** : la proposition y sera recopiée seulement après validation.

La géométrie seule ne prouve rien : elle s'ajoute à l'examen visuel de la section 8.

**Résultat attendu.** Au moins les groupes `{1001, 1020}` et `{1005, 1012}`, signalés « FUITE dans le découpage actuel », puis un lot de test de 5 cas (1 à 0,45 mm, 4 à 0,625 mm) qui ne contient **aucun** de ces quatre cas. Normalement, `sform/qform` n'affiche jamais `0/0`. À examiner ensemble, s'ils apparaissent :
- une ligne « ATTENTION » (groupe de plus de 2 cas, ou paire proche à cheval sur test et entraînement dans la proposition) ;
- le cadre de points d'exclamation **« ORIGINE NON FIABLE »** à la toute fin ;
- un « ARRÊT » : épaisseur de coupe imprévue (ni 0,45 ni 0,625 mm), 3e axe qui n'est pas l'axe des coupes, ou une épaisseur sans assez de patients à un seul examen pour remplir ses places (par exemple aucun à 0,45 mm). Dans ce cas, rien n'est écrit.

**Ensuite.** Copie **toute** la sortie de la cellule, y compris le contenu JSON affiché à la fin, et envoie-la dans la conversation. La reconversion au format nnU-Net viendra seulement après validation du nouveau découpage.

In [ ]:
DONNEES = '/content/drive/MyDrive/poc_ct/mmwhs'
SORTIE = '/content/drive/MyDrive/poc_ct'
!python scripts/03b_regrouper_patients.py --donnees "$DONNEES" --sortie "$SORTIE"